# Water Quality Index (WQI) estimation for Indian rivers
**How to run:** (1) upload this notebook to Colab, (2) Runtime → Run all, (3) when the upload cell asks, choose `water_dataX.csv`. All tables, figures and `results.json` are produced by the code below.

## Step 0: Upload the data file

In [ ]:
import os
if not os.path.exists('water_dataX.csv'):
    try:
        from google.colab import files
        print('Choose water_dataX.csv')
        files.upload()
    except ImportError:
        raise FileNotFoundError('Put water_dataX.csv in the same folder as this notebook')
print('Data file found:', os.path.exists('water_dataX.csv'))

## Imports and settings

In [ ]:
"""
Water Quality Index (WQI) estimation for Indian rivers - Kaggle "Indian water quality data".
Run:  python run_experiments.py   (needs water_dataX.csv in the same folder)
Creates: out/results.json and out/fig_*.png  (every number in the report comes from here)
"""
import json, os, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split, cross_val_score
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, confusion_matrix)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
os.makedirs("out", exist_ok=True)
plt.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6,
                     "ytick.labelsize": 6, "legend.fontsize": 5.8, "axes.spines.top": False, "axes.spines.right": False})
W = 3.3

## Step 1: Load and clean the data

In [ ]:
raw = pd.read_csv("water_dataX.csv", encoding="latin-1")
raw.columns = ["code", "loc", "state", "temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc", "year"]
PARAMS = ["temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc"]
NAMES = {"temp": "Temp", "do": "DO", "ph": "pH", "cond": "Conductivity", "bod": "BOD", "nitrate": "Nitrate+nitrite-N",
         "fc": "Fecal coliform", "tc": "Total coliform"}
R = {"n_raw": int(len(raw))}
for c in PARAMS:  # 'NAN' and blank strings become missing
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw["code"] = raw["code"].astype(str).str.strip().replace({"NAN": np.nan, "nan": np.nan})
R["missing_pct"] = {c: round(float(raw[c].isna().mean() * 100), 1) for c in PARAMS}
R["dup_rows"] = int(raw.duplicated().sum()); raw = raw.drop_duplicates().reset_index(drop=True)
bad_ph = (raw.ph < 0) | (raw.ph > 14)
R["ph_invalid"] = int(bad_ph.sum()); R["ph_invalid_max"] = float(raw.ph[bad_ph].max())
raw.loc[bad_ph, "ph"] = np.nan          # pH must lie between 0 and 14
R["years"] = [int(raw.year.min()), int(raw.year.max())]
R["n_code_missing"] = int(raw.code.isna().sum()); R["n_stations"] = int(raw.code.nunique())
R["rows_by_year"] = {int(k): int(v) for k, v in raw.year.value_counts().sort_index().items()}

## Step 2: Compute the Water Quality Index (WQI)

In [ ]:
STD = {"ph": 8.5, "do": 5.0, "bod": 3.0, "cond": 2250.0, "nitrate": 10.0, "fc": 500.0, "tc": 500.0}   # standard S_i
IDEAL = {"ph": 7.0, "do": 14.6}                                                                       # ideal values V_ideal (others 0)
WQI_P = list(STD)
wts = {k: 1 / v for k, v in STD.items()}; tot = sum(wts.values()); wts = {k: v / tot for k, v in wts.items()}
R["std"] = STD; R["weights"] = {k: float(v) for k, v in wts.items()}
cc = raw.dropna(subset=WQI_P).copy().reset_index(drop=True)      # complete cases for the 7 WQI parameters
R["n_complete"] = int(len(cc)); R["temp_missing_cc"] = int(cc.temp.isna().sum())
Q = pd.DataFrame(index=cc.index)
Q["ph"] = 100 * (cc.ph - IDEAL["ph"]).abs() / (STD["ph"] - IDEAL["ph"])
Q["do"] = 100 * (IDEAL["do"] - cc["do"]) / (IDEAL["do"] - STD["do"])
for k in ["bod", "cond", "nitrate", "fc", "tc"]: Q[k] = 100 * cc[k] / STD[k]
Q = Q.clip(0, 100)                                                # sub-index limited to 0-100
cc["wqi"] = sum(Q[k] * wts[k] for k in WQI_P)
cc["cls"] = np.select([cc.wqi <= 50, cc.wqi <= 75], [0, 1], 2)    # 0 good/excellent, 1 poor, 2 very poor
CLASSES = ["Good/Excellent (<=50)", "Poor (51-75)", "Very poor (>75)"]
R["wqi_desc"] = {k: float(v) for k, v in cc.wqi.describe().items()}
R["class_counts"] = {CLASSES[i]: int((cc.cls == i).sum()) for i in range(3)}
R["n_excellent"] = int((cc.wqi <= 25).sum())
R["share_q_capped"] = {k: float((100 * (cc[k] if k not in ("ph", "do") else np.nan) / STD[k] > 100).mean()) if k not in ("ph", "do") else None for k in WQI_P}
R["bod_over_std_pct"] = float((cc.bod > STD["bod"]).mean() * 100); R["do_below_std_pct"] = float((cc["do"] < STD["do"]).mean() * 100)
R["fc_over_std_pct"] = float((cc.fc > STD["fc"]).mean() * 100); R["tc_over_std_pct"] = float((cc.tc > STD["tc"]).mean() * 100)

## Step 3: Features and chronological split (train 2005-2012, test 2013-2014)

In [ ]:
SKEWED = ["cond", "bod", "nitrate", "fc", "tc"]
FEATS = {"A": PARAMS, "B": ["temp", "do", "ph", "cond"]}   # A = all measured parameters, B = quick field measurements only
def feats(df, cols):
    X = df[cols].copy()
    for c in SKEWED:
        if c in X: X[c] = np.log1p(X[c].clip(lower=0))      # fixed log(1+x) transform: no statistics learned from data
    return X
cc = cc.sort_values("year", kind="stable").reset_index(drop=True)
tr = cc[cc.year <= 2012].reset_index(drop=True); te = cc[cc.year >= 2013].reset_index(drop=True)
R["split"] = {"n_train": len(tr), "n_test": len(te), "train_years": [int(tr.year.min()), int(tr.year.max())], "test_years": [int(te.year.min()), int(te.year.max())],
              "test_station_seen_pct": float(te.code.isin(tr.code).mean() * 100), "test_code_missing": int(te.code.isna().sum()),
              "wqi_mean_train": float(tr.wqi.mean()), "wqi_mean_test": float(te.wqi.mean()),
              "cls_train": [int((tr.cls == i).sum()) for i in range(3)], "cls_test": [int((te.cls == i).sum()) for i in range(3)]}
tscv = TimeSeriesSplit(n_splits=5)

def reg_m(y, p): return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))), "R2": r2_score(y, p)}
def pipe(model, scale=True):
    st = [("imp", SimpleImputer(strategy="median"))]
    if scale: st.append(("sc", StandardScaler()))
    return Pipeline(st + [("m", model)])

## Step 4: Regression models (estimate the WQI)

In [ ]:
reg_models = {
    "Linear regression": (pipe(LinearRegression()), {}),
    "Ridge regression": (pipe(Ridge()), {"m__alpha": [0.1, 1, 10, 100]}),
    "k-NN": (pipe(KNeighborsRegressor()), {"m__n_neighbors": [5, 15, 31], "m__weights": ["uniform", "distance"]}),
    "SVR (RBF)": (pipe(SVR()), {"m__C": [1, 10, 100], "m__epsilon": [0.5, 2]}),
    "Decision tree": (pipe(DecisionTreeRegressor(random_state=SEED), False), {"m__max_depth": [3, 5, 8, 12], "m__min_samples_leaf": [5, 10, 20]}),
    "Random forest": (pipe(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), False), {"m__max_depth": [5, 10, None], "m__min_samples_leaf": [1, 5, 10]}),
    "Gradient boosting": (pipe(GradientBoostingRegressor(random_state=SEED), False), {"m__n_estimators": [100, 300], "m__learning_rate": [0.05, 0.1], "m__max_depth": [2, 3]}),
}
def clean_bp(bp): return {k.replace("m__", ""): (None if v is None else float(v) if isinstance(v, float) else v if isinstance(v, str) else int(v)) for k, v in bp.items()}

ytr, yte = tr.wqi, te.wqi
stn = tr.groupby("code").wqi.mean()
base_station = te.code.map(stn).fillna(ytr.mean())
R["reg"] = {"baselines": {"Mean baseline": reg_m(yte, np.full(len(yte), ytr.mean())), "Station-mean baseline": reg_m(yte, base_station)}, "sets": {}}
best_store = {}; te_store = {}
for sname, cols in FEATS.items():
    Xtr, Xte = feats(tr, cols), feats(te, cols)
    Sx = {"n_features": len(cols), "models": {}}; preds = {}
    for name, (p, grid) in reg_models.items():
        if grid:
            gs = GridSearchCV(clone(p), grid, cv=tscv, scoring="neg_root_mean_squared_error").fit(Xtr, ytr)
            est, cvr, bp = gs.best_estimator_, -gs.best_score_, clean_bp(gs.best_params_)
        else:
            est = clone(p).fit(Xtr, ytr); bp = {}
            cvr = -cross_val_score(clone(p), Xtr, ytr, cv=tscv, scoring="neg_root_mean_squared_error").mean()
        pr = est.predict(Xte); preds[name] = pr
        m = reg_m(yte, pr); m["CV_RMSE"] = float(cvr); m["best_params"] = bp
        Sx["models"][name] = m; best_store[(sname, name)] = est
    best = min(Sx["models"], key=lambda k: Sx["models"][k]["RMSE"]); Sx["best_model"] = best
    Sx["best_vs_mean_pct"] = 100 * (1 - Sx["models"][best]["RMSE"] / R["reg"]["baselines"]["Mean baseline"]["RMSE"])
    Sx["best_vs_station_pct"] = 100 * (1 - Sx["models"][best]["RMSE"] / R["reg"]["baselines"]["Station-mean baseline"]["RMSE"])
    Xa, ya = feats(cc, cols), cc.wqi                                       # random-split check
    Xr, Xs, yr_, ys_ = train_test_split(Xa, ya, test_size=0.3, random_state=SEED)
    Sx["random_split_RMSE_best"] = float(np.sqrt(mean_squared_error(ys_, clone(best_store[(sname, best)]).fit(Xr, yr_).predict(Xs))))
    pi = permutation_importance(best_store[(sname, best)], Xte, yte, n_repeats=30, random_state=SEED, scoring="neg_root_mean_squared_error")
    Sx["perm_importance"] = {f: float(v) for f, v in sorted(zip(cols, pi.importances_mean), key=lambda t: -t[1])}
    rf = best_store[(sname, "Random forest")]
    Sx["rf_importance"] = {f: float(v) for f, v in sorted(zip(cols, rf.named_steps["m"].feature_importances_), key=lambda t: -t[1])}
    te_store[sname] = (yte.values, preds[best])
    R["reg"]["sets"][sname] = Sx
R["reg"]["primary_set"] = "B"

## Step 5: Classification of the WQI category

In [ ]:
def clf_m(y, p): return {"Acc": accuracy_score(y, p), "MacroF1": f1_score(y, p, average="macro", zero_division=0)}
ctr, cte = tr.cls, te.cls
mode_tr = int(ctr.mode()[0]); stm = tr.groupby("code").cls.agg(lambda s: int(s.mode()[0]))
R["clf"] = {"baselines": {"Majority class": clf_m(cte, np.full(len(cte), mode_tr)),
                          "Station-majority": clf_m(cte, te.code.map(stm).fillna(mode_tr).astype(int))}, "sets": {}}
clf_models = {
    "Logistic regression": (pipe(LogisticRegression(max_iter=3000)), {"m__C": [0.01, 0.1, 1, 10]}),
    "k-NN": (pipe(KNeighborsClassifier()), {"m__n_neighbors": [5, 15, 31]}),
    "Decision tree": (pipe(DecisionTreeClassifier(random_state=SEED), False), {"m__max_depth": [3, 5, 8, 12], "m__min_samples_leaf": [5, 10, 20]}),
    "Random forest": (pipe(RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), False), {"m__max_depth": [5, 10, None], "m__min_samples_leaf": [1, 5, 10]}),
    "Gradient boosting": (pipe(GradientBoostingClassifier(random_state=SEED), False), {"m__n_estimators": [100, 300], "m__learning_rate": [0.05, 0.1], "m__max_depth": [2, 3]}),
}
cm_store = {}
for sname, cols in FEATS.items():
    Xtr, Xte = feats(tr, cols), feats(te, cols); Sx = {"models": {}}
    for name, (p, grid) in clf_models.items():
        gs = GridSearchCV(clone(p), grid, cv=tscv, scoring="f1_macro").fit(Xtr, ctr)
        est = gs.best_estimator_; pr = est.predict(Xte)
        m = clf_m(cte, pr); m["CV_MacroF1"] = float(gs.best_score_); m["best_params"] = clean_bp(gs.best_params_)
        Sx["models"][name] = m; cm_store[(sname, name)] = confusion_matrix(cte, pr, labels=[0, 1, 2])
    Sx["best_model"] = max(Sx["models"], key=lambda k: Sx["models"][k]["MacroF1"])
    Sx["confusion_best"] = cm_store[(sname, Sx["best_model"])].tolist()
    R["clf"]["sets"][sname] = Sx

## Step 6: Relationships between parameters and WQI

In [ ]:
cm = cc[PARAMS + ["wqi"]].corr(method="spearman")
R["spearman_wqi"] = {c: float(cm.loc[c, "wqi"]) for c in PARAMS}
R["spearman_do_bod"] = float(cc[["do", "bod"]].corr(method="spearman").iloc[0, 1])
R["median_by_year"] = {int(k): float(v) for k, v in cc.groupby("year").wqi.median().items()}

## Step 7: Figures

In [ ]:
def save(fig, name): fig.savefig(f"out/{name}.png", dpi=300, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(W, 1.9))
ms = pd.Series(R["missing_pct"]).sort_values()
ax.barh([NAMES[k] for k in ms.index], ms.values, color="#1f3a68"); ax.set_xlabel("Missing values (%)")
for i, v in enumerate(ms.values): ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=5.5)
save(fig, "fig_missing")

fig, ax = plt.subplots(figsize=(W, 2.7)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 10.2)
steps = [("Raw data", f"water_dataX.csv, {R['n_raw']} river records, 2003-2014"),
         ("Cleaning", "'NAN' to missing, drop duplicate, pH outside 0-14 to missing"),
         ("Compute WQI", "weighted arithmetic index from 7 parameters (complete cases)"),
         ("Split by year", "train 2005-2012, test 2013-2014"),
         ("Pipeline + CV", "impute, standardise, 7 regressors / 5 classifiers, time-series CV"),
         ("Evaluation", "MAE, RMSE, R2 | accuracy, macro-F1 vs mean and station baselines")]
for i, (t, s2) in enumerate(steps):
    y = 9.9 - i * 1.68
    ax.add_patch(FancyBboxPatch((0.2, y - 1.3), 9.6, 1.3, boxstyle="round,pad=0.02", fc="#eef2f8", ec="#1f3a68", lw=0.6))
    ax.text(0.45, y - 0.42, f"{i+1}  {t}", fontsize=6.3, weight="bold", va="center")
    ax.text(0.45, y - 0.93, s2, fontsize=5.0, va="center", color="#333")
    if i < len(steps) - 1: ax.annotate("", xy=(5, y - 1.62), xytext=(5, y - 1.3), arrowprops=dict(arrowstyle="-|>", lw=0.6, color="#1f3a68"))
save(fig, "fig_pipeline")

fig, ax = plt.subplots(figsize=(W, 2.1))
yrs = sorted(cc.year.unique()); ax.boxplot([cc.wqi[cc.year == y] for y in yrs], showfliers=False, medianprops=dict(color="#d98324")); ax.set_xticklabels([str(y) for y in yrs])
for t, lab in [(50, "50"), (75, "75")]: ax.axhline(t, ls=":", c="r", lw=0.7); ax.text(len(yrs) + 0.55, t, lab, color="r", fontsize=5.5, va="center")
ax.axvline(len(tr.year.unique()) + 0.5, c="grey", lw=0.8); ax.text(len(tr.year.unique()) + 0.6, cc.wqi.max() - 1, "test ->", fontsize=5.5, color="grey")
ax.set_ylabel("WQI"); ax.tick_params(axis="x", rotation=45)
save(fig, "fig_wqi_year")

cl = [NAMES[c] for c in PARAMS] + ["WQI"]
fig, ax = plt.subplots(figsize=(W, 3.0)); im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cl))); ax.set_xticklabels(cl, rotation=90); ax.set_yticks(range(len(cl))); ax.set_yticklabels(cl)
for i in range(len(cl)):
    for j in range(len(cl)): ax.text(j, i, f"{cm.values[i, j]:.2f}".replace("0.", ".").replace("-.", "-."), ha="center", va="center", fontsize=4.3)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=5)
for s_ in ("top", "right"): ax.spines[s_].set_visible(True)
save(fig, "fig_corr")

fig, axs = plt.subplots(1, 2, figsize=(W, 1.95), sharey=True)
for ax, s, ttl in zip(axs, ["A", "B"], ["Set A: all parameters", "Set B: field parameters"]):
    yt, yp = te_store[s]; ax.scatter(yt, yp, s=4, alpha=.5, color="#1f3a68"); lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], c="#d98324", lw=1); m_ = R["reg"]["sets"][s]["models"][R["reg"]["sets"][s]["best_model"]]
    ax.set_title(f"{ttl}\n{R['reg']['sets'][s]['best_model']}\nRMSE {m_['RMSE']:.1f}, R$^2$ {m_['R2']:.2f}", fontsize=5.6); ax.set_xlabel("Observed WQI")
axs[0].set_ylabel("Predicted WQI"); fig.tight_layout(); save(fig, "fig_obs_pred")

fig, axs = plt.subplots(1, 2, figsize=(W, 1.9))
for ax, s, c in zip(axs, ["A", "B"], ["#1f3a68", "#d98324"]):
    imp = R["reg"]["sets"][s]["perm_importance"]; ks = list(imp)[:6][::-1]
    ax.barh([NAMES[k] for k in ks], [imp[k] for k in ks], color=c); ax.set_title(f"Set {s}", fontsize=6.5); ax.set_xlabel("RMSE increase", fontsize=5.8)
fig.tight_layout(); save(fig, "fig_importance")

json.dump(R, open("out/results.json", "w"), indent=1, default=float)

## Step 8: Save results and print the summary

In [ ]:
def pr(d): return {a: (round(b, 3) if isinstance(b, float) else b) for a, b in d.items()}
print({k: R[k] for k in ["n_raw", "dup_rows", "ph_invalid", "ph_invalid_max", "years", "n_code_missing", "n_stations", "n_complete"]})
print("missing", R["missing_pct"]); print("weights", {k: round(v, 4) for k, v in R["weights"].items()})
print("wqi", pr(R["wqi_desc"]), "classes", R["class_counts"], "excellent", R["n_excellent"])
print("over std %: bod", round(R["bod_over_std_pct"], 1), "do below", round(R["do_below_std_pct"], 1), "fc", round(R["fc_over_std_pct"], 1), "tc", round(R["tc_over_std_pct"], 1))
print("split", R["split"]); print("spearman wqi", {k: round(v, 2) for k, v in R["spearman_wqi"].items()}, "do-bod", round(R["spearman_do_bod"], 2))
print("median by year", {k: round(v, 1) for k, v in R["median_by_year"].items()})
print("\nREG baselines"); [print(" ", k, pr(v)) for k, v in R["reg"]["baselines"].items()]
for s, Sx in R["reg"]["sets"].items():
    print("\nSET", s, Sx["n_features"])
    for k, v in Sx["models"].items(): print(f"  {k:20s}", pr(v))
    print("  best", Sx["best_model"], "vs mean %.1f%% vs station %.1f%% random-split RMSE %.2f" % (Sx["best_vs_mean_pct"], Sx["best_vs_station_pct"], Sx["random_split_RMSE_best"]))
    print("  perm", {k: round(v, 2) for k, v in list(Sx["perm_importance"].items())[:6]}); print("  rf", {k: round(v, 3) for k, v in list(Sx["rf_importance"].items())[:6]})
print("\nCLF baselines"); [print(" ", k, pr(v)) for k, v in R["clf"]["baselines"].items()]
for s, Sx in R["clf"]["sets"].items():
    print("\nCLF SET", s)
    for k, v in Sx["models"].items(): print(f"  {k:20s}", pr(v))
    print("  best", Sx["best_model"], Sx["confusion_best"])

## Show the figures

In [ ]:
from IPython.display import Image, display
for n in ['fig_missing','fig_pipeline','fig_wqi_year','fig_corr','fig_obs_pred','fig_importance']:
    display(Image(f'out/{n}.png', width=500))

## Download all outputs (optional)

In [ ]:
import shutil
shutil.make_archive('wqi_outputs', 'zip', 'out')
try:
    from google.colab import files
    files.download('wqi_outputs.zip')
except ImportError:
    print('Saved wqi_outputs.zip')